In [10]:
import os
import glob
import torch
import pandas as pd
from sdv.metadata import Metadata
from sdv.sequential import PARSynthesizer

# 1. Settings
WINDOW_SIZE = 150       # Adjust to match your ML model (100-200 is ideal)
EPOCHS = 100             # Enough to see real convergence; bump to 100+ if still dropping
NUM_SYNTH_WINDOWS = 50  # How many synthetic windows to generate

# Bypass the cuDNN RNN crash while keeping GPU active
torch.backends.cudnn.enabled = False

# 2. Load all 8 CSVs and slice into non-overlapping windows
csv_files = sorted(glob.glob("./data_clean/*.csv"))

windows = []
window_id = 1

for file_path in csv_files:
    temp_df = pd.read_csv(file_path)
    
    # Slice in non-overlapping chunks (drops any short leftover tail)
    for start in range(0, len(temp_df) - WINDOW_SIZE + 1, WINDOW_SIZE):
        chunk = temp_df.iloc[start : start + WINDOW_SIZE].copy()
        chunk["series_id"] = window_id
        
        # Reset Timer(ms) to start at 0.0 for each window
        chunk["Timer(ms)"] = chunk["Timer(ms)"] - chunk["Timer(ms)"].iloc[0]
        
        windows.append(chunk)
        window_id += 1

combined_df = pd.concat(windows, ignore_index=True)
print(f"Created {window_id - 1} windows of {WINDOW_SIZE} rows each ({len(combined_df)} total rows).")

# 3. Build and validate metadata
metadata = Metadata.detect_from_dataframes(data={"time_series": combined_df})
metadata.update_column(table_name="time_series", column_name="series_id", sdtype="id")
metadata.set_sequence_key(table_name="time_series", column_name="series_id")
metadata.set_sequence_index(table_name="time_series", column_name="Timer(ms)")
metadata.validate()

# 4. Train and save
synthesizer = PARSynthesizer(metadata, epochs=EPOCHS, verbose=True)
synthesizer.fit({"time_series": combined_df})

os.makedirs("./.snapshots", exist_ok=True)
synthesizer.save("./.snapshots/par_synthesizer_w_200.pkl")

# 5. Generate and export synthetic windows
synthetic_data = synthesizer.sample_sequences(
    num_sequences=NUM_SYNTH_WINDOWS,
    sequence_length=WINDOW_SIZE
)

if isinstance(synthetic_data, dict):
    synthetic_data = synthetic_data["time_series"]

os.makedirs("./data_features/synth", exist_ok=True)
synthetic_data.to_csv("./data_features/synth/synthetic_time_series_w_200.csv", index=False)

print("Done! Saved synthetic_time_series.csv")

Created 2643 windows of 150 rows each (396450 total rows).


/home/akka/Documents/vscode_projects/SemesterProjectFall2026/.venv/lib/python3.12/site-packages/sdv/single_table/base.py:136: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
/home/akka/Documents/vscode_projects/SemesterProjectFall2026/.venv/lib/python3.12/site-packages/sdv/metadata/metadata.py:1226: UserWarning: The metadata lists columns in a different order than the data. This may result in the synthetic data having a different order.
Affected tables: 'time_series'.
  warnings.warn(


OutOfMemoryError: CUDA out of memory. Tried to allocate 116.00 MiB. GPU 0 has a total capacity of 3.73 GiB of which 40.19 MiB is free. Process 8404 has 3.21 GiB memory in use. Including non-PyTorch memory, this process has 476.00 MiB memory in use. Of the allocated memory 349.83 MiB is allocated by PyTorch, and 46.17 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)